In [1]:
from langchain_groq import ChatGroq

In [3]:
llm = ChatGroq(
    temperature=0, 
    groq_api_key='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', 
    model_name="llama-3.3-70b-versatile"
)
response = llm.invoke("The first person to land on moon was ...")
print(response.content)

The first person to land on the moon was Neil Armstrong. He stepped out of the lunar module Eagle and onto the moon's surface on July 20, 1969, during the Apollo 11 mission. Armstrong famously declared, "That's one small step for man, one giant leap for mankind," as he became the first human to set foot on the moon.


In [7]:
from langchain_community.document_loaders import WebBaseLoader
loader = WebBaseLoader("https://www.maersk.com/careers/vacancies/wd/Data---Machine-Learning-Engineer_R111678/jt-machine-learning-engineer?gad_source=1&gclid=CjwKCAiAh6y9BhBREiwApBLHC7ZHlv_KFqltzYpXcKcVF8qWVfEv0TV_2_iclt2wgPDxgieKD_LZQRoC8w4QAvD_BwE&gclsrc=aw.ds")
page_data = loader.load().pop().page_content
print(page_data)




Machine Learning Engineer vacancy | R111678 | Maersk


 
              Home
             
              Careers
             
              Vacancies
             Share  
      LinkedIn
     
      Facebook
     
      Twitter
     
      Email
     Copy link 
      WeChat
     Machine Learning Engineer 
      Full time
     
      IN - Bangalore
     
      Posted Today
     Apply now
       Opportunity Machine Learning EngineerMaersk, the world's largest shipping company, is transforming into an industrial digital giant that enables global trade with its land, sea and port assets. We are the digital and software development organization that builds products in the areas of predictive science (forecasting, customer and market analytics), optimization and IoT. This position offers the opportunity to build your engineering career in a data and analytics intensive environment, delivering work that has direct and significant impact on the success of our company. Global data analytics d

In [13]:
from langchain_core.prompts import PromptTemplate

prompt_extract = PromptTemplate.from_template(
    """
        ### SCRAPED TEXT FROM WEBSITE:
        {page_data}
        ### INSTRUCTION:
        The scraped text is from the career's page of a website.
        Your job is to extract the job postings and return them in JSON format containing the 
        following keys: `role`, `experience`, `skills` and `description`.
        Only return the valid JSON.
        ### VALID JSON (NO PREAMBLE):    
        """
)
chain_extract = prompt_extract | llm
res = chain_extract.invoke(input={'page_data': page_data})
type(res.content)

str

In [15]:
from langchain_core.output_parsers import JsonOutputParser

json_parser = JsonOutputParser()
json_res = json_parser.parse(res.content)
json_res

{'role': 'Machine Learning Engineer',
 'experience': '3-5 years',
 'skills': ['Python',
  'TensorFlow',
  'PyTorch',
  'scikit-learn',
  'Cloud platforms (e.g., AWS, GCP, Azure)',
  'Data engineering concepts and tools (e.g., ETL pipelines, data warehousing)',
  'Version control systems (e.g., Git) and CI/CD pipelines'],
 'description': 'We are seeking a highly motivated Machine Learning Engineer to join our growing team. In this role, you will collaborate with data engineers, scientists, and developers to enhance the quality and capabilities of our Machine Learning solutions, including their lifecycle management (MLOps).'}

In [17]:
type(json_res)

dict

In [19]:
import pandas as pd

df = pd.read_csv('my_portfolio.csv')
df

,Techstack,Links
0,"React, Node.js, MongoDB",https://example.com/react-portfolio
1,"Angular,.NET, SQL Server",https://example.com/angular-portfolio
2,"Vue.js, Ruby on Rails, PostgreSQL",https://example.com/vue-portfolio
3,"Python, Django, MySQL",https://example.com/python-portfolio
4,"Java, Spring Boot, Oracle",https://example.com/java-portfolio
5,"Flutter, Firebase, GraphQL",https://example.com/flutter-portfolio
6,"WordPress, PHP, MySQL",https://example.com/wordpress-portfolio
7,"Magento, PHP, MySQL",https://example.com/magento-portfolio
8,"React Native, Node.js, MongoDB",https://example.com/react-native-portfolio
9,"iOS, Swift, Core Data",https://example.com/ios-portfolio


In [21]:
import uuid
import chromadb

client = chromadb.PersistentClient('vectorstore')
collection = client.get_or_create_collection(name="portfolio")

if not collection.count():
    for _, row in df.iterrows():
        collection.add(documents=row["Techstack"],
                       metadatas={"links": row["Links"]},
                       ids=[str(uuid.uuid4())])

In [37]:
links = collection.query(query_texts=['skills'], n_results=2).get('metadatas', [])
links

[[{'links': 'https://example.com/ml-python-portfolio'},
  {'links': 'https://example.com/python-portfolio'}]]

In [41]:
job

{'role': 'Machine Learning Engineer',
 'experience': '3-5 years',
 'skills': ['Python',
  'TensorFlow',
  'PyTorch',
  'scikit-learn',
  'Cloud platforms (e.g., AWS, GCP, Azure)',
  'Data engineering concepts and tools (e.g., ETL pipelines, data warehousing)',
  'Version control systems (e.g., Git) and CI/CD pipelines'],
 'description': 'We are seeking a highly motivated Machine Learning Engineer to join our growing team. In this role, you will collaborate with data engineers, scientists, and developers to enhance the quality and capabilities of our Machine Learning solutions, including their lifecycle management (MLOps).'}

In [39]:
job = json_res
job['skills']

['Python',
 'TensorFlow',
 'PyTorch',
 'scikit-learn',
 'Cloud platforms (e.g., AWS, GCP, Azure)',
 'Data engineering concepts and tools (e.g., ETL pipelines, data warehousing)',
 'Version control systems (e.g., Git) and CI/CD pipelines']

In [45]:
prompt_email = PromptTemplate.from_template(
    """
        ### JOB DESCRIPTION:
        {job_description}
        
        ### INSTRUCTION:
        You are ASH, a business development executive at DRDOT Solutions. DRDOT Solutions is an AI, IOT & Software Consulting company dedicated to facilitating
        the seamless integration of business processes through automated tools. 
        Over our experience, we have empowered numerous enterprises with tailored solutions, fostering scalability, 
        process optimization, cost reduction, and heightened overall efficiency. 
        Your job is to write a cold email to the client regarding the job mentioned above describing the capability of DRDOT Solutions 
        in fulfilling their needs.
        Also add the most relevant ones from the following links to showcase DRDOT Solution's portfolio: {link_list}
        Remember you are ASH, BDE at DRDOT Solutions. 
        Do not provide a preamble.
        ### EMAIL (NO PREAMBLE):
        
        """
)
chain_email = prompt_email | llm
res = chain_email.invoke({"job_description": str(job), "link_list": links})
print(res.content)

Subject: Expert Machine Learning Solutions for Enhanced Business Efficiency

Dear Hiring Manager,

I came across your job posting for a Machine Learning Engineer and was impressed by the role's focus on enhancing the quality and capabilities of your Machine Learning solutions. As a Business Development Executive at DRDOT Solutions, I'd like to introduce you to our company's expertise in AI, IoT, and software consulting, which can help fulfill your requirements.

At DRDOT Solutions, we have a proven track record of empowering enterprises with tailored solutions that foster scalability, process optimization, cost reduction, and heightened overall efficiency. Our team of experts has extensive experience in developing and deploying Machine Learning models using Python, TensorFlow, PyTorch, and scikit-learn. We are well-versed in cloud platforms such as AWS, GCP, and Azure, as well as data engineering concepts and tools like ETL pipelines and data warehousing.

Our capabilities align perfec